# PROD: MGSM Bengali Direct — Qwen3.5-9B Q4_K_M GGUF via lm-eval

**Model:** `psxog/SAHAYAKAI` / `Qwen3.5-9B.Q4_K_M.gguf` (~6GB Q4_K_M)

**Task:** `mgsm_direct_bn` (250 Bengali, direct-answer no-CoT, harness v4, dataset `juletxara/mgsm` bn)

**Backend:** `gguf` via `llama-server` (`base_url=http://127.0.0.1:8081`, completion mode)

**Decoding:** greedy (`do_sample=false, temp=0.0`), `max_gen_toks=256`

**Output:** `/content/mgsm_bn_direct/**/results_*.json` + `samples_mgsm_direct_bn_*.jsonl` (nested hash dir — normal harness layout)

## 1. Setup — Install & GPU check

In [ ]:
!nvidia-smi
!pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 llama-cpp-python
!pip install -q "lm-eval>=0.4.13" huggingface_hub
# Fallback if cu121 404s:
# !pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 llama-cpp-python
!curl -LsSf https://llama.app/install.sh | sh
!export PATH="$HOME/.local/bin:$PATH"; ~/.local/bin/llama --version
!python -c "import llama_cpp; print('GPU offload:', llama_cpp.llama_supports_gpu_offload())"
!lm-eval ls tasks 2>&1 | grep -i "mgsm.*bn"
import sys
print("Python", sys.version)

## 2. Download model

In [ ]:
from huggingface_hub import hf_hub_download
import os

repo_id = "psxog/SAHAYAKAI"
filename = "Qwen3.5-9B.Q4_K_M.gguf"
print(f"Downloading {repo_id}/{filename} ...")
model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(f"Model: {model_path} ({os.path.getsize(model_path)/1e9:.2f} GB)")
os.environ["MODEL_PATH"] = model_path
%env MODEL_PATH={model_path}

## 3. Start llama-server on 8081 (gguf backend is server-based, needs base_url)

In [ ]:
import os
print("MODEL_PATH:", os.environ.get("MODEL_PATH"))
# Binary is ~/.local/bin/llama (new wrapper), NOT bare llama-server; 8080 is often held in Colab, so use 8081.
!pkill -9 -f llama 2>/dev/null; sleep 2; echo "killed old server"
!export PATH="$HOME/.local/bin:$PATH"; nohup ~/.local/bin/llama serve -m "$MODEL_PATH" -c 4096 --port 8081 -ngl -1 --parallel 1 > /tmp/llama8081.log 2>&1 &
!sleep 10; curl -s http://127.0.0.1:8081/health; echo; tail -n 20 /tmp/llama8081.log
# Expect first: 503 Loading model (normal, 9B takes 30-90s). blk.32 unused-tensor warnings are normal Qwen3.5 MTP skip.

## 4. Wait until server ready

In [ ]:
!for i in $(seq 1 12); do echo -n "$i: "; curl -s http://127.0.0.1:8081/health; echo; sleep 10; done
!tail -n 10 /tmp/llama8081.log
# Proceed only when a poll returns {"status":"ok"} + log shows 'model loaded' / 'listening on http://127.0.0.1:8081'.

## 5. Verify task (expect mgsm_direct_bn present)

In [ ]:
!lm-eval ls tasks 2>&1 | grep -i "mgsm.*bn"
!lm-eval validate --tasks mgsm_direct_bn
# Expect: dataset juletxara/mgsm bn, doc_to_text 'প্রশ্ন: '+Q+'\nAnswer:', target answer_number, stops প্রশ্ন:/</s>/<|im_end|>, version 4.

## 6. Smoke-test completions endpoint (must return choices before eval)

In [ ]:
!curl -s -X POST http://127.0.0.1:8081/v1/completions -H "Content-Type: application/json" -d '{"prompt":"1+1=","max_tokens":5}' | head -c 300; echo

## 7. Run eval — Bengali direct (no chat template, greedy, max_gen_toks=256)

In [ ]:
# Server-based gguf backend (harness>=0.4.9): pass base_url, NOT pretrained=...
# No --apply_chat_template / --system_instruction -> completion mode (production benchmark).
!lm_eval run --model gguf \
  --model_args base_url=http://127.0.0.1:8081,max_gen_toks=256,timeout=300 \
  --tasks mgsm_direct_bn \
  --num_fewshot 8 \
  --batch_size 1 \
  --output_path /content/mgsm_bn_direct \
  --log_samples \
  --show_config

## 8. Read results (nested hash dir is normal harness layout)

In [ ]:
import json, glob
res_files = sorted(glob.glob("/content/mgsm_bn_direct/**/results_*.json", recursive=True))
samp_files = sorted(glob.glob("/content/mgsm_bn_direct/**/samples_*.jsonl", recursive=True))
print("results:", res_files)
print("samples:", samp_files)
res = json.load(open(res_files[-1]))
print(json.dumps(res.get("results", res), indent=2)[:3000])
print("versions:", res.get("versions"), "n-shot:", res.get("n-shot"))
lines = open(samp_files[-1]).read().strip().split("\n")
print(f"sample rows: {len(lines)} (expect 500 = 250 docs x 2 filters)")
first = json.loads(lines[0])
print("keys:", list(first.keys()))
print(json.dumps(first, indent=2, ensure_ascii=False)[:1500])


## 9. Download

In [ ]:
from google.colab import files
import glob
for p in sorted(glob.glob("/content/mgsm_bn_direct/**/results_*.json", recursive=True)):
    print(f"Ready: {p}")
# files.download(res_files[-1])  # uncomment: downloads results json
# files.download(samp_files[-1])  # uncomment: downloads samples jsonl (few MB)